# 第6章：几何变换

## 编程实践：相似变换、仿射变换与单应变换

---

## 一、几何变换基础

### 1.1 什么是几何变换？

几何变换是改变图像中**像素位置**的操作。每个像素的坐标按照一定的数学规则进行映射。

```
原图像坐标 (x, y) → 变换矩阵 → 新坐标 (x', y')
```

### 1.2 三种基本变换

#### 相似变换 (Similarity Transform)
保持形状不变，允许**旋转、平移和缩放**。
```
变换矩阵 (2x3):
  | a  b  tx |     | cos(θ)  -sin(θ)  tx |
  | c  d  ty |  =  | sin(θ)   cos(θ)  ty |  × s

其中: θ 是旋转角度, s 是缩放比例, (tx, ty) 是平移量
```

#### 仿射变换 (Affine Transform)
比相似变换更自由，允许**任意线性变换+平移**。保持平行线。
```
变换矩阵 (2x3):
  | a  b  tx |     | a  b  tx |
  | c  d  ty |  →  | c  d  ty |

其中 a, b, c, d, tx, ty 是任意参数
可以实现: 旋转、缩放、剪切、翻转等组合
```

#### 单应变换 (Homography Transform)
最一般的变换，将一个平面映射到另一个平面。需要**4个以上对应点**。
```
变换矩阵 (3x3):
  | h11 h12 h13 |
  | h21 h22 h23 |
  | h31 h32 h33 |

其中 h33 通常设为 1 (共9个参数，8个自由度)
可以实现: 透视变换、视角变换
```

### 1.3 变换类型对比
| 变换类型 | 自由度 | 保持性质 | 典型应用 |
|----------|--------|----------|----------|
| 相似变换 | 4 | 形状 | 简单对齐 |
| 仿射变换 | 6 | 平行性 | 倾斜校正 |
| 单应变换 | 8 | 共线性 | 透视校正 |


## 二、实现要求

> 读取彩色图像，进行相似变换/仿射变换/单应变换（变换矩阵参数自行设定），新建白色底图将原图和变换后的结果保存在底图上（能看出前后位置关系）。除 OpenCV 读写函数外，其余代码手写。


In [ ]:
# 导入库
import cv2
import numpy as np
import math

print(f"OpenCV 版本: {cv2.__version__}")

In [ ]:
# 生成测试图像
import numpy as np

print("正在生成测试图像...")
h, w = 400, 500
img = np.zeros((h, w, 3), dtype=np.uint8)
for y in range(h):
    for x in range(w):
        img[y, x, 0] = int(100 * x / w)
        img[y, x, 1] = int(150 * y / h)
        img[y, x, 2] = int(200 * (x + y) / (w + h))
# 网格线
for i in range(0, w, 50):
    cv2.line(img, (i, 0), (i, h), (200, 200, 200), 1)
for i in range(0, h, 50):
    cv2.line(img, (0, i), (w, i), (200, 200, 200), 1)
# 标记点
points = [(50, 50), (w-50, 50), (50, h-50), (w-50, h-50), (w//2, h//2)]
for pt in points:
    cv2.circle(img, pt, 8, (0, 0, 255), -1)
    cv2.circle(img, pt, 12, (255, 255, 255), 2)
cv2.putText(img, "GEO TRANSFORM", (120, h//2 + 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
cv2.imwrite("geo_image.jpg", img)
print("测试图像已生成: geo_image.jpg")

In [ ]:
def bilinear_interpolation(image, x, y):
    """
    双线性插值 (手写实现)
    根据浮点坐标 (x, y) 计算像素值
    """
    h, w = image.shape[:2]
    
    # 如果坐标超出范围，返回边界值
    if x < 0 or x >= w - 1 or y < 0 or y >= h - 1:
        return None
    
    # 取整数部分和小数部分
    x0 = int(x)
    y0 = int(y)
    x1 = x0 + 1
    y1 = y0 + 1
    
    fx = x - x0  # x 方向的小数部分
    fy = y - y0  # y 方向的小数部分
    
    if len(image.shape) == 3:
        # 彩色图像
        result = np.zeros(3, dtype=np.float64)
        for c in range(3):
            # 四个邻域像素值
            v00 = float(image[y0, x0, c])
            v01 = float(image[y0, x1, c])
            v10 = float(image[y1, x0, c])
            v11 = float(image[y1, x1, c])
            
            # 双线性插值
            top = v00 * (1 - fx) + v01 * fx
            bottom = v10 * (1 - fx) + v11 * fx
            result[c] = top * (1 - fy) + bottom * fy
        
        return np.clip(result, 0, 255).astype(np.uint8)
    else:
        # 灰度图像
        v00 = float(image[y0, x0])
        v01 = float(image[y0, x1])
        v10 = float(image[y1, x0])
        v11 = float(image[y1, x1])
        
        top = v00 * (1 - fx) + v01 * fx
        bottom = v10 * (1 - fx) + v11 * fx
        result = top * (1 - fy) + bottom * fy
        return int(np.clip(result, 0, 255))


def apply_transform_manual(image, transform_matrix, output_size):
    """
    手写几何变换
    使用逆向映射 + 双线性插值
    
    参数:
        image: 输入图像
        transform_matrix: 2x3 或 3x3 变换矩阵
        output_size: 输出尺寸 (width, height)
    """
    out_w, out_h = output_size
    
    # 创建输出图像
    if len(image.shape) == 3:
        result = np.zeros((out_h, out_w, 3), dtype=np.uint8)
    else:
        result = np.zeros((out_h, out_w), dtype=np.uint8)
    
    # 判断是 2x3 还是 3x3 矩阵
    if transform_matrix.shape == (3, 3):
        is_homography = True
    else:
        is_homography = False
    
    # 对输出图像的每个像素进行逆向映射
    for y in range(out_h):
        for x in range(out_w):
            if is_homography:
                # 单应变换: 使用齐次坐标
                denom = transform_matrix[2, 0] * x + transform_matrix[2, 1] * y + transform_matrix[2, 2]
                if abs(denom) < 1e-6:
                    continue
                src_x = (transform_matrix[0, 0] * x + transform_matrix[0, 1] * y + transform_matrix[0, 2]) / denom
                src_y = (transform_matrix[1, 0] * x + transform_matrix[1, 1] * y + transform_matrix[1, 2]) / denom
            else:
                # 仿射/相似变换: 2x3 矩阵
                src_x = transform_matrix[0, 0] * x + transform_matrix[0, 1] * y + transform_matrix[0, 2]
                src_y = transform_matrix[1, 0] * x + transform_matrix[1, 1] * y + transform_matrix[1, 2]
            
            # 使用双线性插值获取源图像像素值
            pixel_val = bilinear_interpolation(image, src_x, src_y)
            
            if pixel_val is not None:
                result[y, x] = pixel_val
    
    return result

In [ ]:
def create_similarity_matrix(angle_deg=0, scale=1.0, tx=0, ty=0):
    """
    创建相似变换矩阵 (2x3)
    angle_deg: 旋转角度 (度)
    scale: 缩放比例
    tx, ty: 平移量
    """
    theta = math.radians(angle_deg)
    cos_t = math.cos(theta) * scale
    sin_t = math.sin(theta) * scale
    
    matrix = np.array([
        [cos_t, -sin_t, tx],
        [sin_t,  cos_t, ty]
    ], dtype=np.float64)
    
    return matrix


def create_affine_matrix(a=1, b=0, tx=0, c=0, d=1, ty=0):
    """
    创建仿射变换矩阵 (2x3)
    可以独立控制每个参数
    """
    matrix = np.array([
        [a, b, tx],
        [c, d, ty]
    ], dtype=np.float64)
    
    return matrix


def create_homography_matrix():
    """
    创建单应变换矩阵 (3x3)
    通过手动设定参数实现透视变换
    """
    # 示例: 透视变换效果
    # h11-h33 需要根据具体场景计算
    matrix = np.array([
        [1.0, 0.1, 0],
        [0.1, 1.0, 0],
        [0.002, 0.001, 1.0]
    ], dtype=np.float64)
    
    return matrix

In [ ]:
def draw_transform_result(original, transformed, title_original, title_transformed):
    """
    在白色底图上显示原图和变换后的结果
    """
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    axes[0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    axes[0].set_title(title_original)
    axes[0].axis('off')
    
    axes[1].imshow(cv2.cvtColor(transformed, cv2.COLOR_BGR2RGB))
    axes[1].set_title(title_transformed)
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

    # 同时保存对比图
    h1, w1 = original.shape[:2]
    h2, w2 = transformed.shape[:2]
    
    max_h = max(h1, h2)
    total_w = w1 + w2 + 10  # 加间隔
    
    # 创建白色底图
    canvas = np.ones((max_h, total_w, 3), dtype=np.uint8) * 255
    
    # 放置原图
    canvas[:h1, :w1] = original
    # 放置变换后图像
    canvas[:h2, w1+10:w1+10+w2] = transformed
    
    return canvas

In [ ]:
# ==================== 读取图像 ====================

img = cv2.imread("geo_image.jpg", cv2.IMREAD_COLOR)

if img is not None:
    h, w = img.shape[:2]
    print(f"读取图像成功! 尺寸: {w}x{h}")
else:
    print("读取图像失败!")

In [ ]:
# ==================== 实践1: 相似变换 ====================
if img is not None:
    print("\n" + "="*50)
    print("实践1: 相似变换")
    print("="*50)
    
    # 参数设置
    angle = 30      # 旋转角度 (度)
    scale = 0.8     # 缩放比例
    tx = 50         # x 方向平移
    ty = 30         # y 方向平移
    
    print(f"参数: 旋转={angle}°, 缩放={scale}, 平移=({tx}, {ty})")
    
    # 创建相似变换矩阵
    sim_matrix = create_similarity_matrix(angle, scale, tx, ty)
    print(f"\n相似变换矩阵:\n{sim_matrix}")
    
    # 应用变换
    print(f"\n正在应用相似变换...")
    new_w = int(w * scale) + abs(tx) + 100
    new_h = int(h * scale) + abs(ty) + 100
    sim_result = apply_transform_manual(img, sim_matrix, (new_w, new_h))
    
    # 保存结果
    cv2.imwrite("similarity_result.jpg", sim_result)
    print(f"已保存: similarity_result.jpg")
    
    # 可视化
    canvas = draw_transform_result(
        img, sim_result,
        'Original Image',
        f'Similarity Transform\n(rotate={angle}°, scale={scale})'
    )
    cv2.imwrite("similarity_comparison.jpg", canvas)
    print("对比图已保存: similarity_comparison.jpg")

In [ ]:
# ==================== 实践2: 仿射变换 ====================
if img is not None:
    print("\n" + "="*50)
    print("实践2: 仿射变换")
    print("="*50)
    
    # 创建仿射变换矩阵
    # 可以实现: 旋转 + 缩放 + 剪切 + 翻转 的任意组合
    # 这里演示: 倾斜 + 缩放 + 平移
    a = 1.2    # x 方向缩放
    b = 0.3    # x 方向剪切
    c = 0.2    # y 方向剪切
    d = 0.8    # y 方向缩放
    tx2 = 30   # x 平移
    ty2 = 40   # y 平移
    
    print(f"参数: a={a}, b={b}, c={c}, d={d}, tx={tx2}, ty={ty2}")
    
    affine_matrix = create_affine_matrix(a, b, tx2, c, d, ty2)
    print(f"\n仿射变换矩阵:\n{affine_matrix}")
    
    # 应用变换
    print(f"\n正在应用仿射变换...")
    affine_w = w + abs(tx2) + 100
    affine_h = h + abs(ty2) + 100
    affine_result = apply_transform_manual(img, affine_matrix, (affine_w, affine_h))
    
    # 保存结果
    cv2.imwrite("affine_result.jpg", affine_result)
    print(f"已保存: affine_result.jpg")
    
    # 可视化
    canvas2 = draw_transform_result(
        img, affine_result,
        'Original Image',
        'Affine Transform'
    )
    cv2.imwrite("affine_comparison.jpg", canvas2)
    print("对比图已保存: affine_comparison.jpg")

In [ ]:
# ==================== 实践3: 单应变换 ====================
if img is not None:
    print("\n" + "="*50)
    print("实践3: 单应变换")
    print("="*50)
    
    # 创建单应变换矩阵 (3x3)
    # 模拟透视变换 (梯形效果)
    h_matrix = create_homography_matrix()
    print(f"单应变换矩阵:\n{h_matrix}")
    
    # 应用变换
    print(f"\n正在应用单应变换...")
    homo_w = w + 100
    homo_h = h + 100
    homo_result = apply_transform_manual(img, h_matrix, (homo_w, homo_h))
    
    # 保存结果
    cv2.imwrite("homography_result.jpg", homo_result)
    print(f"已保存: homography_result.jpg")
    
    # 可视化
    canvas3 = draw_transform_result(
        img, homo_result,
        'Original Image',
        'Homography Transform\n(Perspective)'
    )
    cv2.imwrite("homography_comparison.jpg", canvas3)
    print("对比图已保存: homography_comparison.jpg")
    
    # 总结输出
    print("\n" + "="*50)
    print("所有几何变换完成!")
    print("="*50)
    print("\n生成的文件:")
    print("  - similarity_result.jpg  (相似变换结果)")
    print("  - similarity_comparison.jpg (对比图)")
    print("  - affine_result.jpg  (仿射变换结果)")
    print("  - affine_comparison.jpg (对比图)")
    print("  - homography_result.jpg  (单应变换结果)")
    print("  - homography_comparison.jpg (对比图)")

## 三、三种变换对比

### 变换矩阵形式
```
相似变换 (2x3):  | a  b  tx |   其中 a²+b² = scale²
                  | c  d  ty |   c = -b, d = a

仿射变换 (2x3):  | a  b  tx |   a,b,c,d 为任意值
                  | c  d  ty |   无约束条件

单应变换 (3x3):  | h11 h12 h13 |  3x3 矩阵
                  | h21 h22 h23 |  h33 = 1 (归一化)
                  | h31 h32 h33 |
```

### 自由度对比
- **相似变换**: 4 个自由度 (角度、缩放、tx、ty)
- **仿射变换**: 6 个自由度 (a, b, c, d, tx, ty)
- **单应变换**: 8 个自由度 (h11~h32，h33=1)

### 对应点数量
- 相似变换: 需要 2 对对应点
- 仿射变换: 需要 3 对对应点
- 单应变换: 需要 4 对对应点

### 注意事项
1. 逆向映射比正向映射效果更好（不会出现空洞）
2. 双线性插值比最近邻插值效果更平滑
3. 边界处理：超出原图范围的像素通常设为 0（黑色）
4. 输出图像尺寸需要根据变换范围合理设定
5. 单应变换使用齐次坐标 (需要除以 w)
